# Session 1: a quantity that stays finite, and one that does not
**A. The question.** Fluid motion (air in a room, water in a pipe) is described by the
Navier-Stokes equations. One of the seven Clay Millennium Prize problems asks: if you start
a 3-D flow smoothly, does the solution stay smooth for ever, or can some quantity (such as
the speed gradient) become infinite in finite time?

**HONESTY NOTE.** That question is OPEN. Nobody has proved that smooth 3-D solutions
always exist, and nobody has proved that they can fail to exist. This repo does not try to
solve it. It teaches what the problem asks and what is known, and lets you experiment on
simpler models.

**Plan.** (1) see a toy equation that blows up in finite time, (2) see a similar one that
cannot, (3) see how a smooth wave steepens and pushes its energy to small scales.

In [ ]:
import pathlib

import numpy as np

# where figures are saved (works as a script and in Jupyter/Colab)
OUT = (pathlib.Path(__file__).resolve().parent if "__file__" in globals() else pathlib.Path(".")) / "out"

## Step 1: dy/dt = y**p, a toy for "growth feeds on itself"
y is a number that changes with time t. Its rate of change is y raised to the power p.
For p = 2 the bigger y is, the faster it grows, and that is so strong that y reaches
infinity at a finite time. Solving by hand (separate the variables) gives

    y(t) = y0 / (1 - y0 t)        for p = 2, so y is infinite at  t* = 1 / y0.

For a general p > 1 the blow-up time is t* = 1 / ((p - 1) * y0**(p - 1)).
For p = 1 (dy/dt = y) the solution y0 * exp(t) is finite at every finite time.

In [ ]:
def blowup_time(y0, p=2.0):
    """Exact time at which y' = y**p, y(0) = y0 > 0, becomes infinite (p > 1 only)."""
    if p <= 1:
        return float("inf")
    return 1.0 / ((p - 1.0) * y0 ** (p - 1.0))


def exact_y(t, y0, p=2.0):
    """Exact solution of y' = y**p (valid for t below the blow-up time when p > 1)."""
    if p == 1:
        return y0 * np.exp(t)
    return (y0 ** (1.0 - p) - (p - 1.0) * t) ** (-1.0 / (p - 1.0))

## Step 2: check it with a computer
We step y forward with the 4th-order Runge-Kutta method (RK4) and watch it grow. Near the
blow-up y changes extremely fast, so the time step is chosen small in proportion to
how fast y changes: dt = c * y**(1 - p). We stop when y passes a huge number `y_big`.
For p = 2 the exact time at which y equals y_big is 1/y0 - 1/y_big.

In [ ]:
def rk4_step(f, y, dt):
    k1 = f(y)
    k2 = f(y + 0.5 * dt * k1)
    k3 = f(y + 0.5 * dt * k2)
    k4 = f(y + dt * k3)
    return y + dt * (k1 + 2 * k2 + 2 * k3 + k4) / 6.0


def time_to_reach(y0, p, y_big, c=0.02):
    """Integrate y' = y**p with adaptive steps; return (time, y) when y first passes y_big."""
    y, t = float(y0), 0.0
    while y < y_big:
        dt = c * y ** (1.0 - p)
        y = rk4_step(lambda v: v ** p, y, dt)
        t += dt
    return t, y

## Step 3: a smooth wave that steepens
Real fluids move themselves: the speed u carries itself along. A one-line model of this
is the inviscid (no friction) Burgers equation, u_t + u u_x = 0. With u(x, 0) = sin x each
point keeps its speed and moves in a straight line (a "characteristic"):

    u(x, t) = sin(X)   where  x = X + t sin(X).

For t < 1 this is one smooth curve. As t approaches 1 the slope at x = pi tends to minus
infinity: a shock (jump) forms. Session 2 studies this properly. Here we ask what steepening
does to the *energy*: a sine wave has all its energy at wavenumber 1 (one wave per box); a
steep front needs many short waves, so energy moves to small scales. This "cascade to small
scales" is the reason the Navier-Stokes question matters: tiny scales mean huge gradients.

In [ ]:
def characteristic_profile(x, t, iters=60):
    """u(x, t) for the inviscid Burgers equation with u0 = sin x, valid for t < 1."""
    X = np.array(x, dtype=float)
    for _ in range(iters):               # Newton iterations for X + t sin X = x
        X = X - (X + t * np.sin(X) - x) / (1.0 + t * np.cos(X))
    return np.sin(X)


def small_scale_fraction(t, kmin=3, n=256):
    """Fraction of the energy carried by waves with wavenumber >= kmin (n grid points)."""
    x = 2 * np.pi * np.arange(n) / n
    uh = np.fft.fft(characteristic_profile(x, t)) / n
    k = np.fft.fftfreq(n, 1.0 / n)
    e = np.abs(uh) ** 2
    return float(e[np.abs(k) >= kmin].sum() / e.sum())

## Step 4: see it

In [ ]:
if __name__ == "__main__":
    import matplotlib.pyplot as plt

    OUT.mkdir(exist_ok=True)
    t = np.linspace(0, 1.6, 400)
    fig, ax = plt.subplots(figsize=(6, 4))
    for p, col in ((2.0, "tab:red"), (1.5, "tab:orange"), (1.0, "tab:blue")):
        tend = min(1.6, blowup_time(1.0, p) * 0.999)
        tt = t[t < tend]
        ax.plot(tt, exact_y(tt, 1.0, p), color=col, label=f"p = {p}")
    ax.axvline(1.0, color="gray", ls=":")
    ax.set_ylim(0, 30)
    ax.set_xlabel("time t (arbitrary units)")
    ax.set_ylabel("y (arbitrary units)")
    ax.set_title("y' = y^p with y0 = 1: p = 2 is infinite at t = 1")
    ax.grid(alpha=0.3)
    ax.legend()
    fig.tight_layout()
    fig.savefig(OUT / "blowup_vs_smooth.png", dpi=120)
    for y0 in (0.5, 1.0, 2.0):
        tn, _ = time_to_reach(y0, 2.0, 1e6)
        print(f"y0 = {y0}: RK4 reaches 1e6 at t = {tn:.6f}; exact blow-up time {blowup_time(y0):.6f}")

    x = np.linspace(0, 2 * np.pi, 400)
    fig2, (a1, a2) = plt.subplots(1, 2, figsize=(9, 3.6))
    for tt in (0.0, 0.5, 0.9, 0.99):
        a1.plot(x, characteristic_profile(x, tt), label=f"t = {tt}")
    a1.set_xlabel("position x (radians)")
    a1.set_ylabel("speed u (arbitrary units)")
    a1.set_title("A smooth wave steepens")
    a1.grid(alpha=0.3)
    a1.legend()
    ts = np.linspace(0, 0.99, 50)
    a2.plot(ts, [100 * small_scale_fraction(v) for v in ts])
    a2.set_xlabel("time t (arbitrary units)")
    a2.set_ylabel("energy in waves with k >= 3 (percent)")
    a2.set_title("Energy moves to small scales")
    a2.grid(alpha=0.3)
    fig2.tight_layout()
    fig2.savefig(OUT / "steepening.png", dpi=120)
    print(f"small-scale energy: t=0.5 -> {100 * small_scale_fraction(0.5):.1f}%, "
          f"t=0.9 -> {100 * small_scale_fraction(0.9):.1f}%")
    print("saved figures in", OUT)

## Try it
Do the coding tasks, answer the four questions in the README (section F), then run the
checkpoint. Remember: the real 3-D Navier-Stokes question is still open.

## Coding tasks
1. **Easy.** Change the starting value in `lesson.py` to y0 = 3 and predict the blow-up time before running; then compare with the RK4 value.
2. **Stretch.** Write a function that finds the time at which y first passes 1000 for dy/dt = y^2, y0 = 1, and compare it with 1/y0 - 1/1000.
3. **Explain.** Explain in your own words why dy/dt = y has no blow-up but dy/dt = y^2 has, and why a steepening wave moves energy to short waves.

## Assessment
- **A. Conceptual.** In one or two sentences, what does the Navier-Stokes smoothness problem ask, and what is its status?
- **B. Numerical.** For dy/dt = y^2, find the blow-up time for y0 = 4 and for y0 = 0.5. What happens to the time when y0 doubles?
- **C. Graph and simulation interpretation.** On the growth view, which curve reaches the top of the plot first for y0 = 1: p = 2 or p = 1.5? What does p = 1 do?
- **D. Design and optimisation challenge.** Design: you are told y follows dy/dt = y^2 and the model is only trusted until t = 1.5. Find the largest starting value y0 for which no blow-up occurs before t = 1.5, and one reason the answer is only a model result.

## Checkpoint
Run this last cell. It prints PASS when your work is correct.

In [ ]:
import numpy as np


def run_checks(L):
    """L is the lesson module (or notebook namespace). Returns a list of (label, passed)."""
    checks = []
    checks.append(("worked example: y' = y^2 with y0 = 2 blows up at t = 0.5", abs(L.blowup_time(2.0) - 0.5) < 1e-12))
    tn, yn = L.time_to_reach(2.0, 2.0, 1e6)
    checks.append(("RK4 reaches y = 1e6 at the exact crossing time 0.5 - 1/y (within 1e-5)",
                   abs(tn - (0.5 - 1.0 / yn)) < 1e-5 and yn >= 1e6))
    t1, _ = L.time_to_reach(1.0, 2.0, 1e6)
    checks.append(("y0 = 1 gives blow-up time 1 (RK4 within 1e-5)", abs(t1 - 1.0) < 1e-5))
    checks.append(("exact y(t) = y0/(1 - y0 t) agrees: y0 = 1, t = 0.9 gives 10", abs(L.exact_y(0.9, 1.0) - 10.0) < 1e-9))
    checks.append(("doubling y0 halves the blow-up time for p = 2", abs(L.blowup_time(2.0) * 2 - L.blowup_time(1.0)) < 1e-12))
    checks.append(("worked example: y(0.4) = 2 / (1 - 2 * 0.4) = 10 for y0 = 2", abs(L.exact_y(0.4, 2.0) - 10.0) < 1e-9))
    checks.append(("y0 = 4 blows up at 0.25 and y0 = 0.5 at 2", abs(L.blowup_time(4.0) - 0.25) < 1e-12 and abs(L.blowup_time(0.5) - 2.0) < 1e-12))
    checks.append(("p = 1.5 with y0 = 1 blows up at t = 2 (formula)", abs(L.blowup_time(1.0, 1.5) - 2.0) < 1e-12))
    t15, _ = L.time_to_reach(1.0, 1.5, 1e6)
    checks.append(("p = 1.5: RK4 time to reach 1e6 is within 0.01 of 2", abs(t15 - 2.0) < 0.01))
    checks.append(("p = 1 never blows up (infinite blow-up time) and y(10) = e^10 is finite",
                   L.blowup_time(1.0, 1.0) == float("inf") and abs(L.exact_y(10.0, 1.0, 1.0) - np.exp(10.0)) < 1e-6))
    checks.append(("p below 1 never blows up", L.blowup_time(1.0, 0.5) == float("inf")))
    x = np.linspace(0, 2 * np.pi, 7)
    checks.append(("at t = 0 the profile is sin x", np.allclose(L.characteristic_profile(x, 0.0), np.sin(x), atol=1e-12)))
    # hand check: the profile value at x must come from a foot point X with X + t sin X = x
    t = 0.5
    xs = np.linspace(0.1, 6.0, 9)
    u = L.characteristic_profile(xs, t)
    ok = True
    for xv, uv in zip(xs, u):
        cands = [np.arcsin(uv), np.pi - np.arcsin(uv)]
        ok &= any(abs(X + t * np.sin(X) - xv) < 1e-9 or abs(X + 2 * np.pi + t * np.sin(X) - xv) < 1e-9
                  or abs(X - 2 * np.pi + t * np.sin(X) - xv) < 1e-9 for X in cands)
    checks.append(("hand check: every profile value comes from a foot point with X + 0.5 sin X = x", ok))
    f0, f5, f9, f99 = (L.small_scale_fraction(v) for v in (0.0, 0.5, 0.9, 0.99))
    checks.append(("a pure sine wave has no energy at wavenumber >= 3", f0 < 1e-20))
    checks.append(("energy at small scales grows as the front steepens (t = 0, 0.5, 0.9, 0.99)", f0 < f5 < f9 < f99))
    checks.append(("about 7 percent of the energy is at k >= 3 when t = 0.9", abs(100 * f9 - 7.0) < 0.5))
    checks.append(("less than 1 percent at t = 0.5", 100 * f5 < 1.0))
    return checks

import types
_ns = types.SimpleNamespace(**{k: v for k, v in globals().items() if not k.startswith('_')})
_ok = True
for _label, _passed in run_checks(_ns):
    print(('PASS  ' if _passed else 'FAIL  ') + _label)
    _ok &= bool(_passed)
print('PASS' if _ok else 'FAIL')
